Interactive Variable Viewer

In [ ]:
dataset = {"train": [1, 2, 3, 4, 5], "val": [6, 7]}
learning_rate = 0.001

Output Scrolling

In [ ]:
for epoch in range(1, 151):
    print(
        f"Epoch {epoch:03d}/150 - Loss: {1.0 / epoch:.4f} - Accuracy: {0.5 + 0.5 * (epoch / 150):.2f}"
    )

Data Management

In [ ]:
from datasets import load_dataset

# 1. Download and load the MRPC subset of the GLUE Benchmark
dataset = load_dataset("nyu-mll/glue", "mrpc")

# 2. Inspect the structure (returns DatasetDcit with 'train', 'validation', and 'test' splits)
print(dataset)

# 3. Inspect the first 5 examples from the training set
print("\n" + "=" * 50)
for i in range(5):
    example = dataset["train"][i]
    print(f"Example {i + 1}:")
    print(f"  Sentence 1: {example['sentence1']}")
    print(f"  Sentence 2: {example['sentence2']}")
    print(f"Is Paraphrase? {'Yes (1)' if example['label'] == 1 else 'No(0)'}")

Streaming datasets

In [ ]:
import time
from datasets import load_dataset

print("Connecting to stream (allenai/c4)...")
# streaming=True creates an IterableDataset without downloading the 800 GB
streamed_c4 = load_dataset("allenai/c4", "en", split="train", streaming=True)

# Count how many rows we can pull in 10 seconds
start_time = time.time()
count = 0

print("Pulling rows from the stream for 10 seconds...")
for example in streamed_c4:
    count += 1
    if time.time() - start_time >= 10:
        break

elapsed = time.time() - start_time
print(
    f"\nDone! Processed {count:,} examples in {elapsed:.2f} seconds ({count / elapsed:.1f} examples/sec)."
)

# Print a preview of the last streamed text
print("\nSample streamed text snippet:")
print(example["text"][:200] + "...")

Parquet vs CSV

In [ ]:
import os
from datasets import Dataset, load_dataset

# 1. Grab MRPC train dataset
train_ds = load_dataset("nyu-mll/glue", "mrpc", split="train")

# 2. Export to CSV and Parquet
train_ds.to_csv("mrpc_train.csv")
train_ds.to_parquet("mrpc_train.parquet")

# 3. Compare file sizes on disk
csv_size = os.path.getsize("mrpc_train.csv")
parquet_size = os.path.getsize("mrpc_train.parquet")
print(f"CSV Size: {csv_size:>10,} bytes ({csv_size / 1024:.1f} KB)")
print(f"Parquet Size: {parquet_size:>10,} bytes ({parquet_size / 1024:.1f} KB)")
print(f"--> Parquet is {csv_size / parquet_size:.2f}x smaller than CSV\n")

# 4. Reload from Parquet and verify integrity
reloaded_ds = Dataset.from_parquet("mrpc_train.parquet")
print(f"Original Row Count: {len(train_ds)}")
print(f"Reloaded Row Count: {len(reloaded_ds)}")
print(f"Columns match? {train_ds.column_names == reloaded_ds.column_names}")

# Clean up temporary test files
os.remove("mrpc_train.csv")
os.remove("mrpc_train.parquet")

Reproducible 70 / 15 / 15 Data Splits

In [ ]:
from datasets import load_dataset

# load the MRPC dataset
dataset = load_dataset("nyu-mll/glue", "mrpc", split="train")
total_rows = len(dataset)
print(f"Total raw dataset rows: {total_rows}")

# first split: 70% train, 30% temporary holdout
first_split = dataset.train_test_split(test_size=0.3, seed=42)
train_set = first_split["train"]
temp_set = first_split["test"]

# second split: 30% validation set equally in half for validation and test (15% each)
second_split = temp_set.train_test_split(test_size=0.5, seed=42)
val_set = second_split["train"]
test_set = second_split["test"]

# inspect the sizes
print("\n" + "=" * 45)
print(f"Train set:      {len(train_set):>5} rows ({len(train_set)/total_rows:.1%})")
print(f"Validation set: {len(val_set):>5} rows ({len(val_set)/total_rows:.1%})")
print(f"Test set:       {len(test_set):>5} rows ({len(test_set)/total_rows:.1%})")
print("=" * 45)

# verify no data was lost
assert len(train_set) + len(val_set) + len(test_set) == total_rows
print("Data integrity check passed: No rows lost during splitting.")


# show deterministic reproducibility (first row of test set)
print(f"\nSeed test row: '{test_set[0]['sentence1'][:60]}...'")